## Import Liblary

In [8]:
pip install imbalanced-learn

In [63]:
# Liblary general
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Liblary ML
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,recall_score, f1_score, roc_auc_score)
from imblearn.over_sampling import SMOTE

## Data Loading

In [2]:
df = pd.read_csv('/content/credit_cleaned_data.csv')

In [3]:
df.head()

,SK_ID_CURR,TARGET,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,...,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,EXT_SOURCE_2,EXT_SOURCE_3,REGION_RATING_CLIENT,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AGE,YEARS_EMPLOYED
0,100002,1,M,0,202500.0,406597.5,24700.5,351000.0,Secondary,Single,...,Y,House / apartment,0.262949,0.139376,2.0,0.0,0.0,1.0,25,1
1,100003,0,F,0,270000.0,1293502.5,35698.5,1129500.0,Higher education,Married,...,N,House / apartment,0.622246,0.535276,1.0,0.0,0.0,0.0,45,3
2,100004,0,M,0,67500.0,135000.0,6750.0,135000.0,Secondary,Single,...,Y,House / apartment,0.555912,0.729567,2.0,0.0,0.0,0.0,52,0
3,100006,0,F,0,135000.0,312682.5,29686.5,297000.0,Secondary,Civil marriage,...,Y,House / apartment,0.650442,0.535276,2.0,0.0,0.0,0.0,52,8
4,100007,0,M,0,121500.0,513000.0,21865.5,513000.0,Secondary,Single,...,Y,House / apartment,0.322738,0.535276,2.0,0.0,0.0,0.0,54,8


## Split feature

In [36]:
X = df.drop(
    columns=[
        'SK_ID_CURR',
        'TARGET',
        'CNT_CHILDREN',
        'CNT_FAM_MEMBERS',
        'AMT_ANNUITY',
        'FLAG_OWN_CAR'
    ]
)

y = df['TARGET']

In [35]:
df['YEARS_EMPLOYED'].describe()

,YEARS_EMPLOYED
count,9692.000000
mean,182.380004
std,379.710154
min,0.000000
25%,2.000000
50%,6.000000
75%,15.000000
max,1000.000000


## One hot encoding

In [37]:
X = pd.get_dummies(
    X,
    drop_first=True
)

## Train test split

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## SMOTE

In [39]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [40]:
y_train_smote.value_counts()

,count
TARGET,
0,7149
1,7149


## Modelling

In [53]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(
    X_train_smote,
    y_train_smote
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [54]:
thresholds = [0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5]

for threshold in thresholds:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    recall = recall_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    print(
        f"Threshold: {threshold}"
    )

    print(
        f"Recall: {recall:.3f}"
    )

    print(
        f"Precision: {precision:.3f}"
    )

    print(
        f"F1: {f1:.3f}"
    )

    print("-"*30)

Threshold: 0.1
Recall: 0.556
Precision: 0.131
F1: 0.213
------------------------------
Threshold: 0.15
Recall: 0.464
Precision: 0.160
F1: 0.238
------------------------------
Threshold: 0.2
Recall: 0.377
Precision: 0.190
F1: 0.253
------------------------------
Threshold: 0.25
Recall: 0.278
Precision: 0.190
F1: 0.226
------------------------------
Threshold: 0.3
Recall: 0.205
Precision: 0.211
F1: 0.208
------------------------------
Threshold: 0.35
Recall: 0.172
Precision: 0.228
F1: 0.196
------------------------------
Threshold: 0.4
Recall: 0.146
Precision: 0.247
F1: 0.183
------------------------------
Threshold: 0.45
Recall: 0.093
Precision: 0.230
F1: 0.132
------------------------------
Threshold: 0.5
Recall: 0.046
Precision: 0.179
F1: 0.074
------------------------------


In [61]:
y_prob = model.predict_proba(X_test)[:,1]

threshold = 0.10

y_pred = (
    y_prob >= threshold
).astype(int)

## Evaluations

In [62]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1 Score:",
    f1_score(y_test, y_pred)
)

print(
    "ROC AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

Accuracy: 0.6224858174316658
Precision: 0.1222366710013004
Recall: 0.6225165562913907
F1 Score: 0.20434782608695654
ROC AUC: 0.6902454923922544


In [64]:
joblib.dump(model, "xgboost_model.pkl")
print("Model berhasil disimpan!")

Model berhasil disimpan!
